# 🔃 Desafio de Projeto - ETL com Python e IA Generativa

Contexto: Como um cientista de dados de um grande banco recebi a tarefa de envolver clientes de maneira mais personalizada. Para isso usaremos uma IA Generativa para gerar mensagens de marketing personalizadas para cada cliente.

1) Para isso teremos uma planilha (.csv) com uma lista de IDs de usuários do banco.
2) Com os dados dessa planilha, iremos consumir o endpoint ```GET http://santander-api:8080/users/{id}``` (API) onde obtiremos os dados de cada cliente.
3) Com esses dados, usaremos a API do Gemini(gratuita no momento) para gerar a mensagem de marketing personalizada para cada cliente.
    - Essa mensagem deve enfatizar a importância dos investimentos.
4) E com essa mensagem criada, vamos ennviar de volta ao endpoint ```PUT http://santander-api:8080/users/{id}``` (API) atualizando a lista "news" de cada usuário.

In [ ]:
# URL da API
# Repositório da API modificado: https://github.com/MarceloJSSantos/santander-dev-week-2023-api-copia-pipeline-dados
api_url = "http://santander-api:8080"

## 📥 Extract

1) Extrair a lista de IDs a partir do arquivo .csv.
2) Para cada ID, fazer a requisição GET e obter os dados do cliente bancário

In [ ]:
# carregar os dados dos clientes
import pandas as pd
df = pd.read_csv("../dados/ids-clientes.csv")
clientes_id = df["UserID"].tolist()
# print(clientes_id)

In [ ]:
import requests
# import json

def obter_dados_cliente(cliente_id):
    response = requests.get(f"{api_url}/users/{cliente_id}")
    return response.json() if response.status_code == 200 else None

clientes = [cliente for cliente_id in clientes_id if (cliente := obter_dados_cliente(cliente_id)) is not None]
# print(json.dumps(clientes, indent=2))


## ⚙️ Transform

1) Faz uso de uma IA Generativa (Gemini por ser gratuita) para gerar mensagens de marketing para cada cliente

In [ ]:
# Instalar a biblioteca google-generativeai
# biblioteca instalada na imagem do devcontainer, mas caso queira instalar localmente, use o comando abaixo:
# %pip install -U google-genai

In [ ]:
import os
from google import genai

# O método os.getenv busca a variável no sistema operacional (container)
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    raise ValueError("A chave API_KEY não foi encontrada nas variáveis de ambiente!")

client = genai.Client(api_key=api_key)

# listar os modelos disponíveis
# modelos = client.models.list()
# for model in modelos:
#     print(f"ID: {model.name} | Display Name: {model.display_name}")

In [ ]:
import json

# Configurações específicas para cada modelo (RPM e RPD)
CONFIG_MODELOS = {
    "gemini-2.5-flash-lite": {"rpm": 5, "rpd": 20},
    "gemini-3.1-flash-lite-preview": {"rpm": 15, "rpd": 500},
}

def gerar_mensagens_em_lote(lista_clientes, modelo):
    # Extraímos apenas os nomes para economizar tokens
    nomes = [c.get('name', 'Cliente') for c in lista_clientes]
    
    prompt = f"""
    Atuando como um especialista em marketing bancário, crie mensagens personalizadas 
    para cada um dos clientes abaixo sobre a importância dos investimentos.
    
    Regras:
    1. Máximo de 200 caracteres por mensagem.
    2. Use emojis de dinheiro e tom engajador.
    3. Retorne EXCLUSIVAMENTE um objeto JSON no formato:
       {{"mensagens": [ {{"nome": "NOME", "texto": "MENSAGEM"}}, ... ]}}
    
    Clientes: {", ".join(nomes)}
    """

    try:
        response = client.models.generate_content(
            model=f'{modelo}',
            contents=prompt,
            config={'response_mime_type': 'application/json'} # Força o retorno em JSON
        )
       
        # Retornamos os dados E os metadados de uso
        return json.loads(response.text), response.usage_metadata
    except Exception as e:
        print(f"Erro: {e}")
        return None, None

In [ ]:
import time
import datetime

class AdvancedQuotaManager:
    def __init__(self, model_name):
        # Busca a config no dicionário. Se não achar, usa um padrão seguro.
        config = CONFIG_MODELOS.get(model_name, {"rpm": 1, "rpd": 10})
        
        self.model_name = model_name
        self.rpm_limit = config["rpm"]
        self.rpd_limit = config["rpd"]
        
        # Controles de estado
        self.used_today = 0
        self.last_reset = datetime.date.today()
        self.calls_this_minute = 0
        self.minute_start_time = time.time()

    def check_and_register(self, response_metadata=None):
        current_time = time.time()
        
        # Reset de Minuto
        if current_time - self.minute_start_time > 60:
            self.calls_this_minute = 0
            self.minute_start_time = current_time
        
        # Reset de Dia
        if datetime.date.today() > self.last_reset:
            self.used_today = 0
            self.last_reset = datetime.date.today()

        self.used_today += 1
        self.calls_this_minute += 1
        
        total_tokens = response_metadata.total_token_count if response_metadata else 0

        # print(f"\n--- 📊 MONITOR [{self.model_name}] ---")
        # print(f"📅 RPD: {self.used_today}/{self.rpd_limit}")
        # print(f"⏱️ RPM: {self.calls_this_minute}/{self.rpm_limit}")
        # print(f"🔢 Tokens: {total_tokens}")
        # print("------------------------------")

In [ ]:
import random
import json

# # --- Inicialização ---
# 1. Defina qual modelo vai usar
MODELO_ATUAL = "gemini-3.1-flash-lite-preview"
tracker = AdvancedQuotaManager(MODELO_ATUAL) 
icon_list = ["💰", "📈", "💹", "💸", "📊", "🏗️"]

# 1. Chamada da função (Note que agora capturamos DOIS retornos: resultado e metadados)
# Passamos o MODELO_ATUAL para a função também, para manter a consistência
resultado_lote, metadados = gerar_mensagens_em_lote(clientes, modelo=MODELO_ATUAL)

if resultado_lote:
    # 2. Registra o uso (RPM, RPD e TPM) usando os metadados capturados
    tracker.check_and_register(metadados)
    
    # 3. Transformamos a lista do JSON em um dicionário para busca rápida
    # Usamos .get('mensagens', []) para evitar erros se a chave não existir
    mensagens_dict = {item['nome']: item['texto'] for item in resultado_lote.get('mensagens', [])}
    
    # 4. Distribui as mensagens para cada cliente na sua lista local
    for cliente in clientes:
        nome = cliente.get('name')
        texto_marketing = mensagens_dict.get(nome)
        
        if texto_marketing:
            cliente['news'].append({
                "icon": random.choice(icon_list),
                "description": texto_marketing
            })
            print(f"✅ Sucesso: Mensagem adicionada para {nome}")
        else:
            # Caso o Gemini tenha ignorado algum cliente no JSON
            print(f"⚠️ Aviso: O modelo não gerou mensagem para o cliente {nome}")
else:
    # Caso a função tenha retornado (None, None) devido a erro ou falta de cota
    print("❌ Erro: Falha ao processar o lote na API ou cota excedida.")

In [ ]:
#listar os clientes com as mensagens geradas
# print(json.dumps(clientes, indent=2, ensure_ascii=False))

## 📤 Load

1) Com os dados enriquecidos, atualizamos a lista news de cada usuário

In [ ]:
def atualiza_cliente(cliente):
    response = requests.put(f"{api_url}/users/{cliente['id']}", json=cliente)
    return True if response.status_code == 200 else False

for cliente in clientes:
    sucesso = atualiza_cliente(cliente)
    if sucesso:
        print(f"✅ Cliente {cliente['name']} atualizado com sucesso!")
    else:
        print(f"❌ Falha ao atualizar o cliente {cliente['name']}.")

